# シミュレーション計算実行用ノートブック

## 1. 今回の実験の説明

In [ ]:
DESCRIPTION = '''
python bindingの機能を使ってlammpsをlumeから実行する実験
'''
ISSUE_NO = ''
EXEC_NAME = 'Box2'
RUN_SCRIPT = "calcLammps"

PREV_RUNID = '238804f679ea46818daf380f2bfe3b0b'
RESTART_FROM = 'annealed.poly+water'

MLFLOW_EXP_TYPE = "LumeLammpsDev2"

USE_MPI = True
MPI_NP = 4

## 2. シミュレーションパラメータ

In [ ]:
import importlib, json
sim = importlib.import_module(f'scripts.{RUN_SCRIPT}')

sim_params = sim.default_prams

## 必要なら適宜修正する
sim_params["data_file"] = 'data.poly+water'
sim_params["input_file"] = 'in.poly+water2'
sim_params["log_file"] = f'{EXEC_NAME}.txt'
sim_params["data_dir"] = f'data/{EXEC_NAME}'
sim_params["run_steps"] = 5000
sim_params["thermo_step"] = 50

print(json.dumps(sim_params, indent=4, ensure_ascii=False))

dump_files = ["dump.poly2", "dump.water2"]
snapshots = ""
restart_files = []

## 3. mlflow変数

In [ ]:
import mlflow
import os
from time import strftime, gmtime

In [ ]:
ROOT = os.getenv("HOME")

## 1台構成の時
#MLFLOW_TRACKING_URI = f"sqlite:///{ROOT}/mlruns/mlflow.db"
#MLFLOW_STORAGE = f"file://{ROOT}/mlstorage"
### S3 bucket を指定する場合
MLFLOW_STORAGE = f"s3://{os.getenv('S3STORAGEBUCKET', 'my-mlflow-artifact-s3-bucket')}/mlstorage/"

## mlflow serverのIPを指定
MLFLOW_TRACKING_URI = "http://localhost:5000"


## github, backlogなどでチケット管理をしている場合はそのBASE URLを設定
ISSUE_BASE_URL = 'https://xxxxx/'

### dump fileの圧縮に使うコマンド（pixz があれば推奨）
ARCHIVE_COMMAND = "pixz"

###
### mlflow変数　自動設定
###
MYNAME = os.getenv("USER")
#GIT_INFO = gitutils.get_info()
RUN_NAME = EXEC_NAME + strftime("-%Y-%m-%d-%H-%M-%S", gmtime())

if PREV_RUNID != '':
    prev_run = mlflow.get_run(PREV_RUNID)
    prev_experiment_id = prev_run.info.experiment_id
    tracking_uri = mlflow.get_tracking_uri().rstrip("/")
    prev_url = f"{tracking_uri}/#/experiments/{prev_experiment_id}/runs/{PREV_RUNID}"
    
    restart = f"（{RESTART_FROM}）" if RESTART_FROM != '' else ''

    FORMER_EXP = f"\n[PREV_RUNID]({prev_url})より派生{restart}"
else:
    FORMER_EXP = ""

ISSUE_NAME = f'\n[{ISSUE_NO}]({ISSUE_BASE_URL}{ISSUE_NO})' if ISSUE_NO != '' else ''

## 4. シミュレーション実行

In [ ]:
###
### mlflow処理開始
###
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow_exp = mlflow.get_experiment_by_name(MLFLOW_EXP_TYPE)
if mlflow_exp is None:
    mlflow_exp_id = mlflow.create_experiment(name=MLFLOW_EXP_TYPE, artifact_location=MLFLOW_STORAGE)
else:
    mlflow_exp_id = mlflow_exp.experiment_id

In [ ]:
mlflow_run = mlflow.start_run(
    experiment_id=mlflow_exp_id,
    run_name=RUN_NAME,
    description=f'{DESCRIPTION}{FORMER_EXP}{ISSUE_NAME}')

print(f"Run ID: {mlflow_run.info.run_id}")

mlflow.set_tag("mlflow.user", MYNAME)
mlflow.set_tag("simulation", EXEC_NAME)
mlflow.set_tag("run_script", RUN_SCRIPT)
#mlflow.log_params({'git_commit': GIT_INFO['commit'], 'git_branch': GIT_INFO['branch']})
#mlflow.log_artifact('git.diff.txt', artifact_path='git_info')

if USE_MPI:
    mlflow.set_tags({
        "USE_MPI": USE_MPI,
        "MPI_NP": MPI_NP,
    })

In [ ]:
np = MPI_NP if USE_MPI else 1

batch_script = f'''#! /bin/sh
#SBATCH -J {RUN_NAME}
#SBATCH -o {RUN_NAME}.out
#SBATCH -e {RUN_NAME}.err
#SBATCH --nodes=1
#SBATCH --ntasks-per-node={np}
#SBATCH -t 00:00:00

export RUN_SCRIPT={RUN_SCRIPT}
export MLFLOW_TRACKING_URI={MLFLOW_TRACKING_URI}
export RUN_NAME={RUN_NAME}
export ARCHIVE_COMMAND={ARCHIVE_COMMAND}
export PREV_RUNID={PREV_RUNID}
export RESTART_FROM={RESTART_FROM}
# JSON 文字列はシェルで空白区切りされないようにクォートする
export SIM_PARAMS='{json.dumps(sim_params)}'
export DUMP_FILES='{json.dumps(dump_files)}'
export SNAPSHOTS={snapshots}
export RESTART_FILES='{json.dumps(restart_files)}'

if [ {USE_MPI} = "True" ];
then
    mpirun -np {np} python3 Run.py {mlflow_run.info.run_id}
else
    python3 Run.py {mlflow_run.info.run_id}
fi

echo $?
'''

import subprocess

# sbatch コマンドに batch_script を標準入力で渡して実行
proc = subprocess.run(
    ["sbatch"],
    input=batch_script,
    text=True,
    capture_output=True
)
print(proc.stdout)
if proc.stderr:
    print(proc.stderr)

JOBN = proc.stdout.replace('Submitted batch job ', '').strip()

In [ ]:
os.system(f'squeue -u {MYNAME}')

In [ ]:
raise Exception("一旦ここで停止")

---
### 5. 以下は中断処理

In [ ]:
os.system(f'scancel {JOBN}')

In [ ]:
mlflow.log_artifact(f'{RUN_NAME}.out', artifact_path='output')
mlflow.log_artifact(f'{RUN_NAME}.err', artifact_path='output')

In [ ]:
artifacts = {
    "dumpfiles": dump_files,
    "snapshots": [snapshots],
    "restarts": restart_files,
}
if "log_file" in sim_params:
    artifacts.update({"log": f'log.{sim_params["log_file"]}'})

sim.store_artifacts(artifacts, mlflow.log_artifact, f"{ARCHIVE_COMMAND} -t", cleanup=True)

In [ ]:
## 異常・中断時の mlflow.end_run()
run_info = mlflow.get_run(mlflow_run.info.run_id)
if run_info.info.lifecycle_stage == "active":
    mlflow.end_run(status='KILLED')
